# Análise de Dados de Jogos da Plataforma Steam

**Introdução à Ciência dos Dados — UFMG — Entrega 2**

Artur Leão · Eloy Maciel · João Pedro · João Victor · Pedro Gabriel

Repositório: https://github.com/pbarruetavena/tp_ciencia_de_dados

O projeto investiga quais fatores estão associados ao desempenho de um jogo na Steam,
relacionando atributos comerciais (preço, gênero, data de lançamento) a métricas de
desempenho (avaliações, tempo de jogo e número estimado de proprietários).

**Base:** [Steam Games Dataset (Daily Updates)](https://www.kaggle.com/datasets/hubertsidorowicz/steam-games-dataset-daily-updates),
versão fixa 83 (definida em `src/config.py`), com dados de ~144 mil jogos.

**Perguntas de pesquisa respondidas neste notebook**, na ordem do relatório:

1. **3.1** Preço vs. teor das avaliações
2. **3.2** Sazonalidade de lançamentos
3. **3.3** Distribuição de proprietários (*long tail*)
4. **3.5** Preço vs. tempo médio jogado

A pergunta **3.4** (previsão da base de jogadores) faz parte da Entrega 3.

## Configuração

Funciona localmente e no Google Colab. A limpeza da base (conversão das faixas de
`estimated_owners`, taxa de avaliações positivas, datas etc.) é feita por
`src.limpeza.limpar()` e é compartilhada por todas as análises.

In [ ]:
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    # Caminho absoluto: rodar a célula de novo não cria clones aninhados
    REPO = "/content/tp_ciencia_de_dados"
    if not os.path.exists(REPO):
        !git clone -q https://github.com/pbarruetavena/tp_ciencia_de_dados.git {REPO}
    else:
        !git -C {REPO} pull -q --ff-only
    %cd {REPO}
    !pip install -q kagglehub
    !python -m src.download_data
    sys.path.insert(0, REPO)
else:
    # Permite importar src/ a partir de notebooks/<pasta>/
    sys.path.insert(0, str(Path.cwd().parents[1]))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from src.config import FIGURAS
from src.dados import carregar_jogos
from src.limpeza import com_reviews_minimas, limpar

jogos = limpar(carregar_jogos())
jogos.shape

In [ ]:
jogos.head()

## 3.1 Preço vs. teor das avaliações

**Pergunta de pesquisa:** É possível encontrar uma relação entre o preço de um jogo e o teor médio das avaliações que ele recebe dos usuários?

**Responsável:** Artur Leão

In [ ]:
# Filtrando jogos com pelo menos 20 avaliações para evitar distorções estatísticas
df_analise = com_reviews_minimas(jogos, n=20).dropna(
    subset=["price", "taxa_positiva"]
)
print(f"Tamanho da base filtrada para análise: {df_analise.shape[0]} jogos")

### Análise exploratória

**Linha de raciocínio:**
* **Expectativa inicial:** Inicialmente, poderíamos hipotetizar que jogos mais caros geram maiores expectativas nos jogadores, o que poderia resultar em avaliações mais rigorosas, e portanto, taxas positivas potencialmente menores. Em contrapartida, jogos gratuitos costumam atrair públicos muito amplos, o que pode gerar volatilidade nas avaliações. Então, é uma dúvida bem válida e legal de se explorar.
* **O que os gráficos mostram:** A visualização de dispersão mostra como a taxa de avaliações positivas (`taxa_positiva`) se distribui ao longo de diferentes faixas de preço (`price`). Além disso, o comparativo entre jogos gratiutos e pagos revela se existe uma diferença estrutural no teor das avaliações entre esses dois modelos.

In [ ]:
# Gráfico 1: dispersão entre Preço e Taxa de Avaliações Positivas (a taxa é um novo campo de análise apóos a limpeza dos dados)
# Limitamos o preço dos jogos até 60 dollares, pois a enorme maioria dos jogos nao passa desse valor, melhorando a visualização
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=df_analise[df_analise["price"] <= 60],
    x="price",
    y="taxa_positiva",
    alpha=0.3,
    color="royalblue",
    ax=ax,
)

ax.set_title("Relação entre Preço e Taxa de Avaliações Positivas")
ax.set_xlabel("Preço (US$)")
ax.set_ylabel("Taxa de Avaliações Positivas")

plt.show()

# Gráfico 2: Boxplot da Taxa Positiva por Status de Preço (gratuito vs pago)
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=df_analise, x="price_status", y="taxa_positiva", palette="Set2", ax=ax
)

ax.set_title("Distribuição da Taxa Positiva: Jogos Gratuitos vs. Pagos")
ax.set_xlabel("Status de Preço")
ax.set_ylabel("Taxa de Avaliações Positivas")

plt.show()

### Teste de hipótese e intervalo de confiança

Para verificar formalmente se existe relação estatística entre o preço e a satisfação dos usuários(medida pela taxa de avaliações positivas), adotamos duas abordagens complementares:
1. **Teste de Correlação de Spearman:** Avalia a existência de relação monotônica entre o preço contínuo e a taxa de avaliações positivas (escolhido por não assumir normalidade nas distribuições).
   * **$H_0$:** Não há correlação entre o preço do jogo e a sua taxa de avaliações positivas ($\rho = 0$).
   * **$H_1$:** Há correlação entre o preço do jogo e a sua taxa de avaliações positivas ($\rho \neq 0$).

2. **Teste de Mann-Whitney U:** Compara as medianas das taxas positivas entre jogos gratuitos e pagos.
   * **$H_0$:** A distribuição da taxa de avaliações positivas é igual entre jogos gratuitos e pagos.
   * **$H_1$:** A distribuição da taxa de avaliações positivas difere entre jogos gratuitos e pagos.

* **Nível de significância:** $\alpha = 0,05$

In [ ]:
# 1) Teste de Correlação de Spearman
preco = df_analise["price"]
taxa = df_analise["taxa_positiva"]

corr_coef, p_valor_corr = stats.spearmanr(preco, taxa)

print(f"--- Correlação de Spearman ---")
print(f"Coeficiente (rho): {corr_coef:.4f}")
print(f"P-valor: {p_valor_corr:.4e}")

#2) Teste de Mann-Whitney U (Gratuito vs pago)
gratuito_taxa = df_analise[df_analise["gratuito"] == True]["taxa_positiva"]
pago_taxa = df_analise[df_analise["gratuito"] == False]["taxa_positiva"]

stat_mann, p_valor_mann = stats.mannwhitneyu(
    gratuito_taxa, pago_taxa, alternative="two-sided"
)

print(f"\n--- Teste de Mann-Whitney U (Free vs Paid) ---")
print(f"Estatística U: {stat_mann:.2f}")
print(f"P-valor: {p_valor_mann:.4e}")

# cálculo da mediana para contexto nos resultados
print(f"\nMediana da taxa positiva (Gratuitos): {gratuito_taxa.median():.4f}")
print(f"Mediana da taxa positiva (Pagos): {pago_taxa.median():.4f}")

**Resultado:** 
* Embora os testes estatísticos tenham apresentado $p$-valores menores que o nível de significância $\alpha = 0,05$ (devido ao grande volume amostral da base de dados), o **coeficiente de correlação de Spearman foi extremamente próximo de zero ($\rho = 0,0822$)**, indicando uma força de correlação desprezível.
* O teste de Mann-Whitney U apontou diferença estatística entre jogos gratuitos e pagos, mas a diferença prática entre as medianas das taxas positivas é mínima (81,8% para pagos contra 79,1% para gratuitos, uma diferença de cerca de 2,6 pontos percentuais).

**Conclusão:** Rejeitamos formalmente $H_0$ estritamente do ponto de vista matemático devido ao tamanho da amostra, mas **concluímos que não há uma relação prática ou relevante entre o preço de um jogo e o teor médio de suas avaliações**. Os usuários avaliam jogos de forma parecida independentemente de terem custado caro, barato ou serem gratuitos.

## 3.2 Sazonalidade de lançamentos

**Pergunta de pesquisa:** Existe sazonalidade no lançamento de jogos? Jogos lançados em determinados meses/trimestres tendem a ter mais proprietários estimados ou maior tempo médio de jogo?

**Responsável:** Eloy Maciel

### Análise exploratória

A pergunta que temos interesse em responder pode ser dividida em duas partes:

a. Sazonalidade de lançamentos: existe um padrão na contagem de jogos lançados por mês/trimestre?

b. Desempenho por período: jogos lançados em certos meses/trimestres têm mais `owners_mid` ou maior `average_playtime_forever`?

#### Busca por sazonalidade

Começamos tentando enteder o lançamento de jogos por ano.

In [ ]:
print(f"NaT em release_date: {jogos["release_date"].isna().sum()}, {(jogos["release_date"].isna().sum()/jogos["release_date"].size)*100:.3f}%")
print("min/max:", jogos["release_date"].min(), jogos["release_date"].max())

contagem_ano = jogos["ano"].value_counts().sort_index()
print(contagem_ano.tail(20))

In [ ]:
tabela = pd.crosstab(jogos["ano"], jogos["mes"])

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    tabela.drop(index="Total", columns="Total", errors="ignore"),
    cmap="viridis",
    annot=True,
    fmt="d",
    linewidths=0.3,
    cbar_kws={"label": "Nº de lançamentos"},
    ax=ax,
)
ax.set(title=f"Lançamentos por ano e mês",
       xlabel="Mês", ylabel="Ano")
plt.tight_layout()
fig.savefig(FIGURAS / "heatmap_ano_mes.png", dpi=150, bbox_inches="tight")

Antes de 2014, a base tem menos de 500 lançamentos por ano, isso torna a série mensal ruidosa e sensível a poucos jogos. Por isso, restringimos a análise a 2014–2025. O ano de 2026 foi excluído por estar incompleto (o snapshot é de outubro/2026), o que geraria meses com contagem artificialmente baixa

In [ ]:
JANELA_INICIO = 2014
JANELA_FIM = 2025
jogos_janela = jogos[(jogos["ano"] >= JANELA_INICIO) & (jogos["ano"] <= JANELA_FIM)].copy()

In [ ]:
por_mes = jogos_janela["mes"].value_counts().sort_index()
por_trim = jogos_janela["trimestre"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
por_mes.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set(title=f"Lançamentos por mês ({JANELA_INICIO}, {JANELA_FIM})", xlabel="Mês", ylabel="Jogos")

por_trim.plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set(title=f"Lançamentos por trimestre ({JANELA_INICIO}, {JANELA_FIM})", xlabel="Trimestre", ylabel="Jogos")
fig.savefig(FIGURAS / "lancamentos_ano_mes.png", dpi=150, bbox_inches="tight")
plt.tight_layout()

O gráfico mostra concentração de lançamentos entre agosto e dezembro, com vale em janeiro. A seguir, verificamos o quanto esse padrão é estável entre os anos, para separar sazonalidade estrutural de ruído de um ano específico.

In [ ]:
prop_por_ano = (
    pd.crosstab(jogos_janela["ano"], jogos_janela["mes"], normalize="index")
)
media_prop = prop_por_ano.mean()  # cada ano pesa igual
print(media_prop.round(3))
print(f"razão max/min: {media_prop.max() / media_prop.min():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(prop_por_ano, cmap="viridis", annot=True, fmt=".2f",
            linewidths=0.3, cbar_kws={"label": "Proporção dos lançamentos do ano"}, ax=ax)
ax.set(title="Distribuição mensal relativa por ano (2014–2025)",
       xlabel="Mês", ylabel="Ano")
fig.savefig(FIGURAS / "distribuicao_mes_ano.png", dpi=150, bbox_inches="tight")

In [ ]:
cv_por_mes = prop_por_ano.std() / prop_por_ano.mean()
print(cv_por_mes.round(3).sort_values(ascending=False))

In [ ]:
std_abs = prop_por_ano.std()
print(std_abs.round(4).sort_values(ascending=False))

O coeficiente de variação (CV) e o desvio-padrão absoluto mostram que maio é o mês mais instável (CV 0,176; DP 0,015) e junho o mais estável (CV 0,064; DP 0,005), seguido de novembro (CV 0,066). Janeiro, o vale em média, também varia bastante (CV 0,166) — ser o menor em média não significa ser o menor em todos os anos. Outubro, o pico, tem CV baixo (0,078), então o padrão de pico é mais consistente que o de vale.

Há um pico em outubro (que permanece alto em novembro), recuo em dezembro e mínimo em janeiro. Uma hipótese é que publishers evitem a temporada de festas por causa da concorrência com AAA, mas que os dados deste trabalho não permitem confirmar. A magnitude é moderada: outubro recebe cerca de 55% mais lançamentos que janeiro

In [ ]:
dias = pd.Series({1:31, 2:28.25, 3:31, 4:30, 5:31, 6:30,
                  7:31, 8:31, 9:30, 10:31, 11:30, 12:31})
taxa_dia = media_prop / dias
taxa_dia = taxa_dia / taxa_dia.sum()

indice = taxa_dia * 12
print(indice.round(4).sort_values(ascending=False))
print(f"razão max/min: {indice.max() / indice.min():.2f}")

In [ ]:
comparacao = pd.DataFrame({
    "media_prop_x12": (media_prop * 12).round(3),
    "indice_dia":     indice.round(3),
})
comparacao["dif_%"] = ((comparacao["indice_dia"] / comparacao["media_prop_x12"] - 1) * 100).round(1)
print(comparacao.sort_values("indice_dia", ascending=False))

O índice ajustado por dias confirma a leitura: outubro concentra 17% mais lançamentos que a média (índice 1,17) e janeiro 25% menos (índice 0,75), com razão pico/vale de 1,55. Como outubro e janeiro têm ambos 31 dias, a razão é a mesma antes e depois do ajuste; a única mudança relevante é fevereiro (+8%, por ter 28,25 dias).

A regressão linear da fatia de cada mês ao longo de 2014–2025, apresentada abaixo, não revela tendência sistemática: nenhum mês apresenta slope significativo após correção para múltiplos testes (Holm). A maior alteração (medida pelo slope) é janeiro (+0,0019/ano), mas não sobrevive à correção. Isso confirma que o padrão sazonal é estável no período, sem deriva

In [ ]:
from scipy.stats import linregress

for m in range(1, 13):
    slope, _, r, p, _ = linregress(prop_por_ano.index, prop_por_ano[m])
    print(f"mês {m:2d}: slope={slope:+.4f}/ano, p={p:.3f}")

In [ ]:
from statsmodels.tsa.seasonal import STL

serie = (jogos_janela.dropna(subset=["release_date"])
         .set_index("release_date").resample("MS").size())
serie = serie.iloc[:-1]

stl = STL(serie, period=12, robust=True).fit()
fig = stl.plot()
fig.savefig(FIGURAS / "decomposicao_stl_lancamentos.png", dpi=150, bbox_inches="tight")

var_r = np.var(stl.resid, ddof=1)
var_rs = np.var(stl.seasonal + stl.resid, ddof=1)
print(f"Força da sazonalidade: {max(0, 1 - var_r/var_rs):.3f}")

Existe sazonalidade nos lançamentos da Steam, moderada e estável. Outubro concentra 17% mais lançamentos que a média e janeiro 25% menos (razão 1,55), já descontando a duração dos meses. O heatmap mostra perfil semelhante em todos os anos, e o CV por mês fica abaixo de 0,18, indicando padrão estrutural. A série temporal confirma força de sazonalidade de 0,53. Esses resultados descrevem quando os jogos são lançados.

#### Desempenho por período

Temos duas escolhas de desenho, antes dos números:

**Trimestre em vez de mês.** Agrupar por trimestre aumenta o N por célula
(4 grupos em vez de 12, \~3× mais jogos por grupo) e reduz ruído de composição
mensal. O padrão da primeira pergunta já está capturado em 4 trimestres (Q4 = out–nov–dez é o pico; Q1 = jan–mar é o vale), então não se perde a informação relevante.

**`average_playtime_forever` é análise secundária por construção.** Após a limpeza, \~93% dos jogos têm esse campo como `NaN` (zero = sem dado de jogadores, não ausência de jogadores). O tamanho N da amostra por trimestre vai despencar e ficar enviesado para jogos com jogadores reais, provavelmente indies grandes e AAAs, que não são representativos dos lançamentos da base. Por isso, o resultado de playtime **não será tratado como achado principal**: é um sinal complementar, com N por grupo reportado e ressalva explícita se o N por trimestre ficar abaixo de um piso (\~5.000).

**Hipóteses para `log_owners`:**
- **H₀:** a distribuição de `log_owners` é igual nos quatro trimestres.
- **H₁:** ao menos um trimestre tem distribuição diferente.
- **Teste:** Kruskal-Wallis (cauda longa, outliers tipo CS:GO/Dota) + OLS com dummies de trimestre e controle por ano, para separar "efeito de trimestre" de "efeito de época"
(Steam cresce muito entre 2014 e 2025, e Q4 concentra lançamentos recentes — sem `C(ano)`, trimestre captura época).
- **α = 0,05.**

In [ ]:
amostra = jogos_janela.dropna(subset=["log_owners", "ano"]).copy()
amostra["trimestre"] = amostra["trimestre"].astype(int)
amostra["ano"] = amostra["ano"].astype(int)

print(f"N janela 2014–2025: {len(jogos_janela):,}")
print(f"N amostra (log_owners + ano): {len(amostra):,} "
      f"({len(amostra)/len(jogos_janela)*100:.1f}%)")
print("\nN por trimestre (owners):")
print(amostra.groupby("trimestre").size().rename("n"))

print("\n--- Checagem de playtime (análise secundária) ---")
amostra_playtime = jogos_janela.dropna(subset=["log_playtime", "ano"]).copy()
print(f"N com log_playtime + ano: {len(amostra_playtime):,} "
      f"({len(amostra_playtime)/len(jogos_janela)*100:.1f}%)")
print("\nN por trimestre (playtime):")
print(amostra_playtime.groupby("trimestre").size().rename("n"))

A amostra de `log_owners` mantém **82,4%** dos lançamentos da janela
(98.862 de 119.992), com N equilibrado entre trimestres (22.873–26.864).
A perda está espalhada entre os trimestres, então a amostra é
representativa dos lançamentos da base.

A amostra de `log_playtime` cai para **8,5%** (10.258 jogos), como previsto
pela limpeza. O N por trimestre (2.190–2.814) é suficiente para
Kruskal-Wallis, mas não para OLS com controle por ano, e a amostra é
enviesada para jogos com jogadores reais. Por isso, playtime entra como análise exploratória (KW + boxplot), com ressalva explícita no fechamento. A análise principal é sobre `log_owners`.

Também tentamos descrever a amostra através de gráficos e tabelas:

In [ ]:
desc = (amostra
        .groupby("trimestre")["log_owners"]
        .agg(n="size", mediana="median", media="mean",
             q25=lambda s: s.quantile(.25),
             q75=lambda s: s.quantile(.75)))
desc["iqr"] = desc["q75"] - desc["q25"]
print(desc.round(3))

In [ ]:
escape = (
    amostra.assign(acima_20k=amostra["owners_mid"] > 20_000)
    .groupby("trimestre")["acima_20k"]
    .mean()
    .mul(100)
    .round(2)
)

print(escape.rename("% acima de 20k donos"))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(escape.index, escape.values, color="steelblue")
ax.set_xticks([1, 2, 3, 4])
ax.set(title="% de jogos que ultrapassam 20k donos, por trimestre (2014–2025)",
       xlabel="Trimestre", ylabel="% acima de 20k")
for b, v in zip(bars, escape.values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.3, f"{v:.1f}%",
            ha="center", fontsize=9)
plt.tight_layout()
fig.savefig(FIGURAS / "grafico_escape.png", dpi=150, bbox_inches="tight")

A tabela descritiva apresentada revela que mediana, Q25 e Q75 são idênticos em todos os trimestres (9,21), com IQR = 0. Isso ocorre porque mais de 75% dos jogos estão no piso da base (0–20k donos). Como as métricas de tendência central não diferenciam os trimestres, resumimos a comparação na taxa de escape, isso é, a proporção de jogos que ultrapassam 20k donos.

O gráfico mostra Q1 = 24,8%, Q2 = 24,8%, Q3 = 24,3% e Q4 = 22,5%. A diferença é modesta (2,3 p.p. entre Q1 e Q4), mas já indica o sentido oposto ao intuitivo.

In [ ]:
from scipy.stats import kruskal

grupos = [g["log_owners"].values for _, g in amostra.groupby("trimestre")]
H, p = kruskal(*grupos)
k = len(grupos)
n = len(amostra)
eps2 = (H - k + 1) / (n - k)  # tamanho de efeito para Kruskal-Wallis

print(f"Kruskal-Wallis: H = {H:.2f}, gl = {k-1}, p = {p:.4g}")
print(f"ε² = {eps2:.4f}  (~variância explicada pelo trimestre)")

In [ ]:
import statsmodels.formula.api as smf

# Modelo 1: ingênuo — confunde trimestre com época
m1 = smf.ols("log_owners ~ C(trimestre)", data=amostra).fit()

# Modelo 2: trimestre + controle por ano — resposta real
m2 = smf.ols("log_owners ~ C(trimestre) + C(ano)", data=amostra).fit()

comp = pd.DataFrame({
    "ingenuo":      m1.params.filter(like="trimestre"),
    "p_ingenuo":    m1.pvalues.filter(like="trimestre"),
    "controlado":   m2.params.filter(like="trimestre"),
    "p_controlado": m2.pvalues.filter(like="trimestre"),
})
print(comp.round(4))
print(f"\nR² m1 = {m1.rsquared:.4f}  |  R² m2 = {m2.rsquared:.4f}")
print(f"ΔR² ao adicionar ano = {m2.rsquared - m1.rsquared:.4f}")

Ao ler o resultado do OLS no modelo ingênuo (só trimestre), o efeito é fraco ($R² = 0,0004$): Q2 ($+0,006$) e Q3 ($-0,008$) não diferem de Q1; apenas Q4
é significativo ($-0,048$). Ao adicionar `C(ano)`, os coeficientes
amplificam em vez de cair: Q3 passa a $-0,054$ (agora significativo)
e Q4 a $-0,091$ (segue significativo). O modelo ingênuo estava, portanto,
mascarando o efeito de trimestre.

O $ΔR²$ de $0,1024$ mostra que é o ano que explica ~10% da variância;
trimestre sozinho explica $0,04\%$. A leitura correta: no mesmo ano, jogos
lançados em Q3/Q4 têm $~5–9\%$ menos donos que jogos de Q1 (coeficientes
$-0,054$ e $-0,091$ em log).

O trimestre tem efeito estatisticamente significativo sobre owners_mid após controlar por ano, mas de magnitude modesta (5–9% no mesmo ano) e contribuição pequena em variância explicada. O sentido
é **oposto ao intuitivo**: Q3/Q4 têm menos donos que Q1. Porém os dois padrões
se encaixam: Q4 concentra mais lançamentos (razão pico/vale 1,55),
mas cada jogo de Q4 tem menos donos em média. Consistente com
competição: mais jogos disputando a mesma atenção → menos donos
por jogo. A sazonalidade de publicação se traduz em diferença de
desempenho negativamente.

O OLS controla o ano, mas não o tempo de
mercado dentro do ano. Q4/2025 teve ~10 meses menos de exposição que
Q1/2025, e `owners_mid` satura no piso (0–20k). Parte do coeficiente
negativo de Q4 pode ser maturação, não sazonalidade.

In [ ]:
grupos_pt = [g["log_playtime"].values
             for _, g in amostra_playtime.groupby("trimestre")]
H_pt, p_pt = kruskal(*grupos_pt)
k_pt = len(grupos_pt)
n_pt = len(amostra_playtime)
eps2_pt = (H_pt - k_pt + 1) / (n_pt - k_pt)

print(f"Kruskal-Wallis (playtime): H = {H_pt:.2f}, p = {p_pt:.4g}")
print(f"ε² = {eps2_pt:.4f}")
print("\nN por trimestre (playtime):")
print(amostra_playtime.groupby("trimestre").size().rename("n"))

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=amostra_playtime, x="trimestre", y="log_playtime",
            color="darkorange", ax=ax)
ax.set(title="log(playtime) por trimestre de lançamento (exploratório)",
       xlabel="Trimestre", ylabel="log(playtime)")
fig.savefig(FIGURAS / "boxplot_log_playtime_trimestre.png", dpi=150, bbox_inches="tight")
plt.tight_layout()

`average_playtime_forever` entrou como **análise secundária**, não como
achado principal, ele serve como sinal complementar, com ressalva explícita.

**Resultado.** Kruskal-Wallis em `log_playtime` rejeita H₀
(H = 8,44, gl = 3, p = 0,038 < 0,05), mas o ε² de 0,0005 indica
**efeito desprezível** — trimestre explica ~0,05% da variância dos
ranks. Com N = 10.258, o p-valor é sensível a diferenças minúsculas:
significância estatística não é relevância prática.

**N por trimestre:** 2.190 (Q1), 2.508 (Q2), 2.746 (Q3), 2.814 (Q4) —
suficiente para o teste, mas a amostra é **enviesada por construção**
(93% dos jogos foram removidos por NaN em playtime; os que sobraram são
sistematicamente jogos com jogadores reais).

**Conclusão.** Não há sinal interpretável de sazonalidade em
`average_playtime_forever`: o teste é estatisticamente significativo
mas o efeito é desprezível, e a amostra não é representativa do
universo de lançamentos. Este resultado **não entra na conclusão
principal da 3.2** — é sinal exploratório, não achado. A análise
principal é sobre `log_owners`.

## 3.3 Distribuição de proprietários (*long tail*)

**Pergunta de pesquisa:** A distribuição de proprietários estimados segue uma "long tail"? Ou seja, poucos jogos concentram a maioria dos jogadores?

**Responsável:** João Pedro Alves Menicucci

In [ ]:
x = jogos["owners_mid"].dropna()
x.shape

### Análise exploratória

O atributo `estimated_owners` vem em faixas (ex.: `"0 - 20000"`); `src.limpeza.limpar()` já converte cada faixa para o ponto médio numérico `owners_mid` e trata `"0 - 0"` como NaN (falta de dado do SteamSpy). Sobram 13 faixas distintas, e a mais baixa (`0 - 20000`) concentra cerca de 80% dos jogos — a base é grosseiramente agregada.

**O que se espera encontrar:** se a distribuição for uma *long tail*, a média fica muito acima da mediana e poucos jogos (os mega-hits) concentram a maioria dos jogadores.

**Como verificar (abordagem IQR):** calculamos os quartis de `owners_mid`, definimos o limite superior de outliers `LS = Q3 + 1.5*IQR` e medimos que fração do total de jogadores está acima dele. Com a maior parte da massa na primeira faixa, esperamos `Q1 = Q3` (IQR = 0) na amostra completa — por isso repetimos o mesmo cálculo na cauda (jogos acima da mediana), onde o IQR é informativo.

In [ ]:
media, dp = x.mean(), x.std(ddof=1)
q1, q2, q3 = x.quantile([0.25, 0.5, 0.75])
iqr = q3 - q1
ls = q3 + 1.5 * iqr

print(f"n = {len(x):,}   média = {media:,.0f}   dp = {dp:,.0f}")
print(f"Q1 = {q1:,.0f}   Q2 (mediana) = {q2:,.0f}   Q3 = {q3:,.0f}")
print(f"IQR = {iqr:,.0f}   LS = Q3 + 1.5*IQR = {ls:,.0f}")

In [ ]:
outliers = x > ls
n_out = int(outliers.sum())
conc_out = x[outliers].sum() / x.sum() * 100

print(f"Outliers superiores (x > LS): {n_out:,} jogos ({outliers.mean() * 100:.1f}%)")
print(f"Concentração: {conc_out:.1f}% de todos os jogadores estimados estão nos outliers")

In [ ]:
cauda = x[x > q2]
q1c, q3c = cauda.quantile([0.25, 0.75])
iqr_cauda = q3c - q1c
ls_cauda = q3c + 1.5 * iqr_cauda

mega = x > ls_cauda
print(f"Cauda (x > {q2:,.0f}): n = {len(cauda):,}")
print(f"Q1 = {q1c:,.0f}   Q3 = {q3c:,.0f}   IQR = {iqr_cauda:,.0f}   LS = {ls_cauda:,.0f}")
print(f"Mega-hits (x > LS da cauda): {int(mega.sum()):,} jogos ({mega.mean() * 100:.1f}%) "
      f"concentram {x[mega].sum() / x.sum() * 100:.1f}% dos jogadores")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(y=x, ax=ax1)
ax1.set_yscale("log")
ax1.set(title="Amostra completa (caixa colapsa: IQR = 0)",
        ylabel="Proprietários (escala log)")

sns.boxplot(y=cauda, ax=ax2)
ax2.set_yscale("log")
ax2.set(title=f"Cauda (x > {q2:,.0f})", ylabel="Proprietários (escala log)")

fig.savefig(FIGURAS / "boxplot_owners.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(x, bins=np.logspace(4, np.log10(x.max()), 40), ax=ax)
ax.set_xscale("log")
ax.set(title="Histograma de proprietários estimados",
       xlabel="Proprietários (escala log)", ylabel="Jogos")
fig.savefig(FIGURAS / "histograma_owners.png", dpi=150, bbox_inches="tight")
plt.show()

**Interpretação dos gráficos:** no boxplot da amostra completa a caixa colapsa na mediana (IQR = 0) mesmo em escala log — os outliers, todos acima da primeira faixa, vão até 150 milhões de proprietários; o boxplot da cauda mostra a caixa de forma legível (Q1 = 35.000, Q3 = 150.000) e a cauda acima dela. O histograma (eixo x em log) cai bruscamente já na primeira faixa e arrasta uma cauda longa à direita.

**Leitura dos resultados:**

- **Visão A (amostra completa):** o IQR dá 0 porque ~80% dos jogos está na faixa `0 - 20000` (Q1 = Q3 = mediana = 10.000), efeito da granularidade das faixas do SteamSpy — que é, em si, evidência de concentração. Com `LS = Q3 + 1.5*IQR = 10.000`, os outliers superiores são ~1/5 dos jogos e concentram ~91% de todos os jogadores estimados.
- **Visão B (cauda, acima da mediana):** aqui o IQR é informativo (Q1 = 35.000, Q3 = 150.000) e `LS = 322.500` separa os mega-hits: ~4,4% dos jogos concentram sozinhos ~79% dos jogadores.

Em qualquer uma das duas visões, poucos jogos concentram a maioria dos jogadores — o padrão de *long tail* que a pergunta de pesquisa previa.

### Teste de hipótese e intervalo de confiança

#### Intervalo de confiança de 95% para a média

Estimamos o IC por **bootstrap**: reamostramos os jogos com reposição B = 10.000 vezes, calculamos a média em cada reamostragem e tomamos os percentis 2,5% e 97,5% dessa distribuição bootstrap.

In [ ]:
B = 10_000
boot = stats.bootstrap(
    (x.to_numpy(),),
    np.mean,
    n_resamples=B,
    batch=200,
    method="percentile",
    confidence_level=0.95,
    random_state=np.random.default_rng(42),
)
ic95 = boot.confidence_interval

print(f"Média = {media:,.0f}   dp = {dp:,.0f}   n = {len(x):,}   B = {B:,} reamostragens")
print(f"IC 95% (bootstrap, percentis): [{ic95.low:,.0f}, {ic95.high:,.0f}]")
print(f"Mediana = {q2:,.0f} -> "
      f"{'abaixo do limite inferior' if q2 < ic95.low else 'dentro do intervalo'}")

**Comparação média vs. mediana:** a mediana (10.000 proprietários, o jogo "típico") fica muito abaixo do limite inferior do IC da média (≈ 80 mil). A média é puxada pelos mega-hits e não representa o jogo típico — evidência estatística robusta de uma distribuição fortemente assimétrica à direita.

#### Teste de aderência — Kolmogorov-Smirnov

Testamos duas candidatas para a forma da distribuição de proprietários:

- **H₀¹:** a distribuição de proprietários estimados é Exponencial.
- **H₀²:** a distribuição de proprietários estimados é Log-normal.
- **H₁:** a distribuição não segue a candidata testada.
- **Teste:** Kolmogorov-Smirnov de uma amostra (`scipy.stats.kstest`) para cada candidata, com parâmetros estimados dos próprios dados (λ = 1/média na Exponencial; μ e σ de `ln(owners_mid)` na Log-normal).
- **Nível de significância:** α = 0,05

In [ ]:
lam = 1 / media
estatistica, p_valor = stats.kstest(x, "expon", args=(0, media))

print(f"lambda = 1 / média = {lam:.3e}")
print(f"Estatística K-S = {estatistica:.4f}   p-valor = {p_valor:.3e}")

In [ ]:
logx = np.log(x)
mu_ln, sigma_ln = logx.mean(), logx.std(ddof=1)
estat_ln, p_ln = stats.kstest(x, "lognorm", args=(sigma_ln, 0, np.exp(mu_ln)))

print(f"mu = {mu_ln:.4f}   sigma = {sigma_ln:.4f}")
print(f"Estatística K-S = {estat_ln:.4f}   p-valor = {p_ln:.3e}")

In [ ]:
t = np.logspace(4, np.log10(x.max()), 400)

fig, ax = plt.subplots(figsize=(8, 5))
sns.ecdfplot(x=x, ax=ax, label="Empírica")
ax.plot(t, stats.expon.cdf(t, scale=media), label="Exponencial")
ax.plot(t, stats.lognorm.cdf(t, sigma_ln, scale=np.exp(mu_ln)), label="Log-normal")
ax.set_xscale("log")
ax.set(title="CDFs teóricas x CDF empírica de proprietários",
       xlabel="Proprietários estimados (escala log)", ylabel="F(x)")
ax.legend()
fig.savefig(FIGURAS / "cdf_owners.png", dpi=150, bbox_inches="tight")
plt.show()

**Resultado:** os dois p-valores são praticamente zero, muito menores que α = 0,05. Rejeitamos H₀¹ e H₀².

**Comparação:** a estatística K-S da Log-normal (≈ 0,46) é bem menor que a da Exponencial (≈ 0,68) — a log-normal descreve melhor a cauda pesada, mas ainda assim não adere.

**Interpretação:** a anatomia dos dados (acumulados em intervalos) dificulta a análise sob Kolmogorov-Smirnov.

#### Teste de aderência — Qui-quadrado (comparação com a Exponencial e a Log-normal)

O K-S compara a CDF empírica com a teórica; o qui-quadrado de aderência compara contagens observadas e esperadas faixa a faixa — mais natural aqui, já que a base só vem em faixas. Testamos as mesmas duas candidatas do K-S:

- **H₀¹:** a distribuição de proprietários estimados é Exponencial (com λ = 1/média).
- **H₀²:** a distribuição de proprietários estimados é Log-normal (com μ e σ estimados de `ln(owners_mid)`).
- **H₁:** a distribuição não segue a candidata testada.
- **Teste:** qui-quadrado de aderência de uma amostra (`scipy.stats.chisquare`) para cada candidata, agrupando os jogos nas faixas da Steam e comparando com a probabilidade teórica em cada faixa. Como os parâmetros são estimados dos dados, descontamos um grau de liberdade por parâmetro (`ddof=1` na Exponencial, `ddof=2` na Log-normal).
- **Nível de significância:** α = 0,05

**Por que ele é adequado:** trabalhamos com contagens por faixa, mantendo frequência esperada ≥ 5 em todas elas (a cauda foi agrupada a partir de 500.000 justamente por isso).

In [ ]:
bordas = np.array([0, 20_000, 50_000, 100_000, 200_000, 500_000, np.inf])
observado = pd.cut(x, bins=bordas).value_counts().sort_index()
esperado_exp = len(x) * np.diff(stats.expon.cdf(bordas, scale=media))
esperado_ln = len(x) * np.diff(stats.lognorm.cdf(bordas, sigma_ln, scale=np.exp(mu_ln)))

quadro = pd.DataFrame({
    "faixa": observado.index.astype(str),
    "observado": observado.to_numpy(),
    "esperado_exp": esperado_exp,
    "esperado_ln": esperado_ln,
})
print(quadro.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nmenor frequência esperada = {min(esperado_exp.min(), esperado_ln.min()):,.0f}")

estat_q_exp, p_q_exp = stats.chisquare(observado.to_numpy(), esperado_exp, ddof=1)
estat_q_ln, p_q_ln = stats.chisquare(observado.to_numpy(), esperado_ln, ddof=2)
print(f"Exponencial: qui-quadrado = {estat_q_exp:,.1f}   gl = {len(observado) - 2}   p-valor = {p_q_exp:.3e}")
print(f"Log-normal:  qui-quadrado = {estat_q_ln:,.1f}   gl = {len(observado) - 3}   p-valor = {p_q_ln:.3e}")

**Resultado:** os dois p-valores são praticamente zero, muito menores que α = 0,05. Rejeitamos H₀¹ e H₀² — em acordo com o K-S.

**Comparação:** a Log-normal tem qui-quadrado bem menor que a Exponencial (≈ 110 mil contra ≈ 276 mil), descrevendo melhor a distribuição, mas ainda assim não adere.

**Interpretação:** a tabela mostra onde cada candidata falha. A Exponencial prevê ~25 mil jogos na primeira faixa, onde há ~97 mil, e despeja massa demais nas intermediárias (prevê ~30 mil entre 50 mil e 100 mil, contra ~5 mil observados). A Log-normal chega mais perto (~72 mil na primeira faixa), mas ainda subestima a concentração inicial, exagera nas faixas intermediárias e subestima a cauda de mega-hits (prevê ~75 jogos acima de 500 mil, contra ~2,5 mil observados). O decaimento real é muito mais brusco no início e a cauda é mais pesada do que qualquer uma das duas.

### Resultado

**Resultado:** ~21% dos jogos (outliers superiores acima de LS = 10.000) concentram ~91% de todos os jogadores estimados; recortando a cauda, ~4,4% dos mega-hits (acima de LS = 322.500) concentram ~79%. A mediana (10.000) fica muito abaixo do IC 95% da média por bootstrap (≈ [80 mil; 95 mil]) e o teste K-S rejeita tanto a Exponencial quanto a Log-normal (p ≈ 0; a log-normal chega mais perto, D ≈ 0,46 contra 0,68), assim como o qui-quadrado de aderência, contra a Exponencial (χ² ≈ 276 mil) e contra a Log-normal (χ² ≈ 110 mil), ambos com p ≈ 0.

**Conclusão:** sim — a distribuição de proprietários estimados é uma *long tail* extrema: poucos jogos concentram a maioria dos jogadores, e nenhuma das duas distribuições clássicas de cauda captura essa concentração.

## 3.5 Preço vs. tempo médio jogado

**Pergunta de pesquisa:** É possível encontrar uma relação entre o preço de um jogo e o tempo médio jogado?

**Responsável:** Pedro Gabriel Barruetavena Vieira

### Análise Exploratória de Dados (EDA)

Investigação inicial dos dados, verificando distribuições, dispersões e padrões.

#### Valores inválidos

In [ ]:
cols_interesse = [
    'price',
    'average_playtime_forever',
    'median_playtime_forever',
]

print('VALORES NULOS')
print(jogos[cols_interesse].isnull().sum())

print('\nESTATÍSTICAS DESCRITIVAS')
print(jogos[cols_interesse].describe())

Existem muitos valores NaN para as variáveis de tempo jogado (problema da base)

In [ ]:
# eliminando NaN, pois é um problema da base: não tem esses dados fornecidos pela steam

jogos_valid = jogos.dropna(
    subset=['price', 'average_playtime_forever', 'median_playtime_forever']
).copy()

print(f'Total de registros válidos para a análise: {len(jogos_valid)}')

In [ ]:
print(f'Tamanho da base válida (jogos_valid): {jogos_valid.shape}')

cols = ['price', 'average_playtime_forever', 'median_playtime_forever']

print('\nPERCENTIS')
print(
    jogos_valid[cols].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

#### Assimetria

A análise abaixo, mostra uma assimetria muito grande nas horas jogadas, dentro da base

In [ ]:
print('\nASSIMETRIA SKEW')
print(jogos_valid[cols].skew())

print('\nPROPORÇÃO DE JOGOS GRATUITOS VS PAGOS')
free_count = (jogos_valid['price'] == 0).sum()
paid_count = (jogos_valid['price'] > 0).sum()
print(f'Jogos Gratuitos (Preço = 0): {free_count} ({free_count/len(jogos_valid)*100:.2f}%)')
print(f'Jogos Pagos (Preço > 0): {paid_count} ({paid_count/len(jogos_valid)*100:.2f}%)')

Assimetria forte pode ser visualizada abaixo

In [ ]:
sns.set_theme(style='whitegrid')

# para ver a assimetria observada acima
plt.figure(figsize=(14, 5))

# sbpt 1 - escala normal
plt.subplot(1, 2, 1)
sns.histplot(
    jogos_valid['average_playtime_forever'],
    bins=50,
    color='steelblue',
    kde=True,
)
plt.title('Tempo de Jogo\n')
plt.xlabel('Tempo Médio')
plt.ylabel('Frequência de Jogos')

# sbpt 2: escala logaritmica
plt.subplot(1, 2, 2)
sns.histplot(
    np.log1p(jogos_valid['average_playtime_forever']),
    bins=50,
    color='darkorange',
    kde=True,
)
plt.title(
    'Tempo de Jogo (log1p)\n'
)
plt.xlabel('Log(Tempo Médio + 1)')
plt.ylabel('Frequência')

plt.tight_layout()
plt.show()

In [ ]:
# --- grafico 2: Relação entre Preço e Tempo de Jogo ---
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=jogos_valid,
    x='price',
    y='average_playtime_forever',
    alpha=0.4,
    color='purple',
)
plt.title(
    'Gráfico de Dispersão: Preço vs. Tempo Médio de Jogo'
)
plt.xlabel('Preço do Jogo ($)')
plt.ylabel('Tempo Médio de Jogo')
plt.show()

#### Estratificação

In [ ]:
bins = [-0.01, 0.00, 9.99, 29.99, 200.0]
labels = [
    'Gratuito ($0)',
    'Baixo ($0.01 - $9.99)',
    'Médio ($10.00 - $29.99)',
    'Alto ($30.00+)',
]
jogos_valid['faixa_preco'] = pd.cut(
    jogos_valid['price'], bins=bins, labels=labels
)

resumo_faixas = (
    jogos_valid.groupby('faixa_preco', observed=False)['average_playtime_forever']
    .agg(
        contagem='count',
        mediana=lambda x: x.median(),
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
    )
    .reset_index()
)
print('RESUMO POR FAIXA DE PREÇO')
print(resumo_faixas)

# boxplot das faixas
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=jogos_valid,
    x='faixa_preco',
    y='average_playtime_forever',
    showfliers=False,
    palette='Set2',
)
plt.title(
    'Distribuição do Tempo Médio de Jogo por Faixa de Preço'
)
plt.xlabel('Faixa de Preço')
plt.ylabel('Tempo Médio de Jogo (Minutos)')
plt.show()

#### Análise de segmentação por gênero

In [ ]:
import ast

def parse_genres(val):
  if isinstance(val, list):
    return val
  if isinstance(val, str):
    try:
      return ast.literal_eval(val)
    except:
      return []
  return []


# explodindo a coluna
listas_generos = jogos_valid['genres'].apply(parse_genres)
todos_generos = listas_generos.explode().dropna().str.strip()

generos_unicos = todos_generos.unique()

print(f'Total de gêneros únicos encontrados: {len(generos_unicos)}')
print('\nTODOS OS GENEROS')
print(todos_generos.value_counts())

Visualizando os generos mais comuns (mais de 300)

In [ ]:
df_generos_explodido = jogos_valid.copy()
df_generos_explodido['genres_list'] = df_generos_explodido['genres'].apply(
    parse_genres
)
df_generos_explodido = df_generos_explodido.explode('genres_list')

# quais generos possuem mais de 300 jogos
contagem_generos = df_generos_explodido['genres_list'].value_counts()
generos_acima_300 = contagem_generos[contagem_generos > 300].index.tolist()

print(f'Gêneros com mais de 300 jogos: {generos_acima_300}')

# filtrando
df_filtrado = df_generos_explodido[
    df_generos_explodido['genres_list'].isin(generos_acima_300)
].copy()

# grafico de dispersão
g = sns.lmplot(
    data=df_filtrado,
    x='price',
    y='average_playtime_forever',
    hue='genres_list',
    col='genres_list',
    col_wrap=3,
    height=4,
    aspect=1.2,
    scatter_kws={'alpha': 0.3},
    line_kws={'color': 'red'},
)

g.set(ylim=(0, 5000), xlim=(0, 100))
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle(
    'Relação Preço vs. Tempo de Jogo'
)
plt.show()

### Teste de Hipótese e Inferência Estatística

- **H₀ (Hipótese Nula):** Não há correlação estatisticamente significativa entre o preço de um jogo e o seu tempo médio jogado.
- **H₁ (Hipótese Alternativa):** Existe uma correlação estatisticamente significativa entre o preço de um jogo e o seu tempo médio jogado
- **Teste estatístico escolhido:** Correlação de Spearman (por ser um teste não paramétrico adequado para analisar a relação entre duas variáveis assimétrics)
- **Nível de significância (α):** 0,05

In [ ]:
from scipy import stats

stat, p_valor = stats.spearmanr(
    jogos_valid['price'], jogos_valid['average_playtime_forever']
)

print(f'Coeficiente de Spearman: {stat:.4f}')
print(f'P-valor: {p_valor:.4e}')

#### Teste de hipótese segmentado

A segmentação por gênero foi aplicada para verificar se a correlação positiva observada entre preço e tempo de jogo se mantém constante ou se é moderada pelas características intrínsecas de cada categoria de jogo, como diferentes nichos possuem dinâmicas de engajamento distintas.

In [ ]:
df_generos_explodido = jogos_valid.copy()
df_generos_explodido['genres_list'] = df_generos_explodido['genres'].apply(
    parse_genres
)
df_generos_explodido = df_generos_explodido.explode('genres_list')

# filtro
contagem_generos = df_generos_explodido['genres_list'].value_counts()
generos_acima_300 = contagem_generos[contagem_generos > 300].index.tolist()

# correlações
resultados_generos = []

for genero in generos_acima_300:
  sub_df = df_generos_explodido[
      df_generos_explodido['genres_list'] == genero
  ].dropna(subset=['price', 'average_playtime_forever'])

  corr, p_val = stats.spearmanr(sub_df['price'], sub_df['average_playtime_forever'])

  alpha = 0.05
  resultados_generos.append({
      'Gênero': genero,
      'qtd': len(sub_df),
      'Spearman': round(corr, 4),
      'P-valor': p_val,
      'significancia': p_val < alpha,
  })

# exibição
df_resumo_generos = pd.DataFrame(resultados_generos)
print('CORRELAÇÃO DE SPERMAN POR GENERO')
print(
    df_resumo_generos.sort_values(by='Spearman', ascending=False).to_string(
        index=False
    )
)

### Conclusão

- **Interpretação dos resultados:** O teste de Spearman aplicado entre o preço e o tempo médio de jogo resultou em um coeficiente de 0,3335 com um p-valor extremamente reduzido. O coeficiente indica uma correlação positiva moderada e monotônica: à medida que o preço do jogo aumenta, o tempo médio despendido pelos jogadores também tende a subir. O p-valor muito baixo valida a significancia estatística do teste.
- **Conclusão:** Rejeita-se a hipótese nula. Resposta à pergunta de pesquisa: Sim, a amostra avaliada indica a existência de uma relação estatística significativa entre o preço de um jogo e o tempo médio jogado. Jogos de maior valor comercial apresentam, em média, maior retenção de tempo por parte dos usuários em comparação aos jogos de menor custo.
- **Análise Complementar:** A desagregação por categorias demonstrou que a correlação positiva entre preço e engajamento se manifesta de forma consistente na maioria dos gêneros principais da plataforma. Apenas o segmento Free To Play não apresentou correlação estatisticamente significativa, evidenciando, talvez, que em jogos gratuitos o modelo de monetização por preço deixa de ser um fator preditivo para o tempo de retenção do usuário.